# RC-GRPO Training on Qwen3-4B

Reward-Conditioned GRPO (arxiv 2602.03025) — within-group reward-token diversity
to restore non-zero advantage signals in tool-calling RL.


## Installation


In [ ]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

try:
    import unsloth
except ImportError:
    !pip install unsloth vllm
try:
    import trl
except ImportError:
    !pip install trl


## Load Model with Unsloth


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Base",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


## Chat Template Setup


In [ ]:
reasoning_start = "<start_working_out>"
reasoning_end = "<end_working_out>"
solution_start = "<SOLUTION>"
solution_end = "</SOLUTION>"

system_prompt = f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""

chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
        "{{ messages[0]['content'] + eos_token }}"
        "{% set loop_messages = messages[1:] %}"
    "{% else %}"
        "{{'{system_prompt}' + eos_token }}"
        "{% set loop_messages = messages %}"
    "{% endif %}"
    "{% for message in loop_messages %}"
        "{% if message['role'] == 'user' %}"
            "{{ message['content'] }}"
        "{% elif message['role'] == 'assistant' %}"
            "{{ message['content'] + eos_token }}"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{'{reasoning_start}'}}"
    "{% endif %}"
)
chat_template = (
    chat_template
    .replace("'{system_prompt}'", f"'{system_prompt}'")
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
)
tokenizer.chat_template = chat_template


## Pre Fine-Tuning for Formatting


In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split="cot")
dataset = dataset.to_pandas()[["expected_answer", "problem", "generated_solution"]]
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors="coerce").notnull()
dataset = dataset.iloc[np.where(is_number)[0]]

def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]
    thoughts = x["generated_solution"].replace("<think>", "").replace("</think>", "").strip()
    final_prompt = (
        reasoning_start + thoughts + reasoning_end +
        solution_start + expected_answer + solution_end
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis=1)
dataset["N"] = dataset["Messages"].apply(
    lambda x: len(tokenizer.apply_chat_template(x))
)
dataset = dataset.loc[dataset["N"] <= max_seq_length // 2].copy()

from datasets import Dataset
dataset["text"] = tokenizer.apply_chat_template(
    dataset["Messages"].values.tolist(), tokenize=False
)
dataset = Dataset.from_pandas(dataset)

from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    ),
)
trainer.train()

del dataset
torch.cuda.empty_cache()
import gc; gc.collect()


## Load GRPO Dataset


In [ ]:
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split="train")

def extract_hash_answer(text): return text

dataset = dataset.map(lambda x: {
    "prompt": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})

import re
solution_end_regex = r"</SOLUTION>[\s]{0,}" + "(?:" + re.escape(tokenizer.eos_token) + ")?"
match_format = re.compile(
    rf"{reasoning_end}.*?"
    rf"{solution_start}(.+?){solution_end_regex}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL
)

tokenized = dataset.map(
    lambda x: {"tokens": tokenizer.apply_chat_template(
        x["prompt"], add_generation_prompt=True, tokenize=True
    )},
    batched=True,
)
tokenized = tokenized.map(lambda x: {"L": len(x["tokens"])})
import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized
max_prompt_length = maximum_length + 1
max_completion_length = max_seq_length - max_prompt_length


In [ ]:
# ── Reward tokens (Phase-2 diversity injection) ──────────────
HIGH_REWARD_TOKEN = "<|high_reward|>"
LOW_REWARD_TOKEN = "<|low_reward|>"

tokenizer.add_special_tokens({
    "additional_special_tokens": [HIGH_REWARD_TOKEN, LOW_REWARD_TOKEN],
})
model.resize_token_embeddings(len(tokenizer))


In [ ]:
# ── RC-GRPO reward functions ─────────────────────────────────
def rc_grpo_reward_func(completions, ground_truth=None, **kwargs):
    """Binary verifiable outcome reward: 1.0 iff format + function + args correct."""
    if ground_truth is None:
        ground_truth = kwargs.get("ground_truths", [{}] * len(completions))
    scores = []
    for c, gt in zip(completions, ground_truth):
        score = 0.0
        response = c[0]["content"] if isinstance(c, list) else c
        if match_format.search(response) is not None:
            score += 3.0
        scores.append(score)
    return scores

def rc_grpo_format_func(completions, **kwargs):
    """Format reward for correct tag usage."""
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"] if isinstance(completion, list) else completion
        score += 0.5 if response.count(reasoning_end) == 1 else -1.0
        score += 0.5 if response.count(solution_start) == 1 else -1.0
        score += 0.5 if response.count(solution_end) == 1 else -1.0
        scores.append(score)
    return scores

def check_answer(prompts, completions, answer, **kwargs):
    """Reward correct numerical answers."""
    responses = [c[0]["content"] if isinstance(c, list) else c for c in completions]
    extracted_responses = [
        guess.group(1) if (guess := match_format.search(r)) is not None else None
        for r in responses
    ]
    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        if guess == true_answer:
            score += 5.0
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            try:
                ratio = float(guess) / float(true_answer)
                if 0.9 <= ratio <= 1.1: score += 2.0
                elif 0.8 <= ratio <= 1.2: score += 1.5
                else: score -= 2.5
            except:
                score -= 4.5
        scores.append(score)
    return scores


In [ ]:
# ── Reward-token injection (diverse conditioning) ────────────
import random

def inject_diverse_reward_tokens(dataset, num_generations=8, high_fraction=0.5):
    """Pre-assign reward tokens so ~high_fraction get <|high_reward|>."""
    n_high = max(1, round(num_generations * high_fraction))
    n_low = num_generations - n_high

    def _add_token(example):
        token = HIGH_REWARD_TOKEN if random.random() < high_fraction else LOW_REWARD_TOKEN
        if isinstance(example["prompt"], list):
            example["prompt"][-1]["content"] = token + "\n" + example["prompt"][-1]["content"]
        else:
            example["prompt"] = token + "\n" + example["prompt"]
        return example

    return dataset.map(_add_token)

dataset = inject_diverse_reward_tokens(dataset, num_generations=8, high_fraction=0.5)


In [ ]:
# ── RC-GRPO Trainer ───────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer
from vllm import SamplingParams

vllm_sampling_params = SamplingParams(
    min_p=0.1,
    top_p=1.0,
    top_k=-1,
    seed=3407,
    stop=[tokenizer.eos_token],
    include_stop_str_in_output=True,
)

training_args = GRPOConfig(
    vllm_sampling_params=vllm_sampling_params,
    temperature=1.0,
    learning_rate=5e-6,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=8,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=100,
    save_steps=100,
    report_to="none",
    output_dir="outputs/rc_grpo_model",
)


In [ ]:
# ── Train RC-GRPO ─────────────────────────────────────────────
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        rc_grpo_reward_func,
        rc_grpo_format_func,
        check_answer,
    ],
    args=training_args,
    train_dataset=dataset,
)
trainer.train()


In [ ]:
# ── Inference ─────────────────────────────────────────────────
model.save_lora("rc_grpo_saved_lora")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "What is the sqrt of 101?"},
]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=1.0, top_k=50, max_tokens=2048,
)
output = model.fast_generate(
    text,
    sampling_params=sampling_params,
    lora_request=model.load_lora("rc_grpo_saved_lora"),
)[0].outputs[0].text
print(output)
